# Project 04 — Shared battery + cabin thermal plant
## 04A → 04G integrated notebook
All values are illustrative/generic unless explicitly marked as a published literature benchmark. No employer data is used.


In [ ]:
from pathlib import Path
import sys, numpy as np, matplotlib.pyplot as plt
HERE=Path.cwd()
if not (HERE/'model.py').exists(): HERE=Path('projects/04-shared-battery-cabin-cooling')
sys.path.insert(0,str(HERE.resolve()))
from shared_cycle import solve_shared_cycle
from compressor_map import operating_point
from heat_exchangers import condenser_condensing_temperature_c
from loads import CabinInputs, BatteryInputs, cabin_cooling_load_kw, battery_heat_kw
from plant import simulate
from fault_cases import FAULTS, run_fault_case, summarize
from validation import compare_to_literature, calibrate_efficiency_to_point, verification_checks


# 04A — Couple shared loads to the refrigerant cycle
The fixed cabin/battery demand and vapor-compression model are solved in one chain.


In [ ]:
base = solve_shared_cycle(20,8,25,5,55)
{k: round(v,3) for k,v in base.items() if isinstance(v,float)}


# 04B — Compressor map / operating envelope
The map is synthetic and generic. Its role is to expose pressure-ratio and speed dependence, not represent a production compressor.


In [ ]:
prs=np.linspace(2,6,50)
for speed in [0.5,0.75,1.0]:
    cap=[operating_point(pr,speed)['capacity_kw'] for pr in prs]
    plt.plot(prs,cap,label=f'speed={speed:.2f}')
plt.xlabel('Pressure ratio [-]'); plt.ylabel('Available capacity [kW]')
plt.title('04B synthetic compressor map'); plt.grid(True,alpha=.3); plt.legend(); plt.show()


# 04C — Condenser heat-rejection limitation
For the same rejection load, reducing fan/air-side effectiveness raises the condensing temperature required by the reduced-order model.


In [ ]:
q=35.0
fans=np.linspace(.3,1.0,40)
tcond=[condenser_condensing_temperature_c(q,45,f) for f in fans]
plt.plot(fans,tcond); plt.xlabel('Fan ratio [-]'); plt.ylabel('Condensing temperature [°C]')
plt.title('04C condenser air-side sensitivity'); plt.grid(True,alpha=.3); plt.show()


# 04D — Physical cabin and battery loads
Cabin load includes envelope, solar, occupants, ventilation and a latent term. Battery heat starts from $I^2R$ plus optional reversible heat.


In [ ]:
cab=cabin_cooling_load_kw(CabinInputs())
bat=battery_heat_kw(BatteryInputs())
cab, bat


# 04E — Transient plant and controls
The plant couples thermal states, load generation, compressor/fan command, capacity allocation and the vapor-compression cycle.


In [ ]:
rows=simulate(minutes=60)
m=np.array([r['minute'] for r in rows])
tb=np.array([r['battery_c'] for r in rows]); tc=np.array([r['cabin_c'] for r in rows])
plt.plot(m,tb,label='Battery'); plt.plot(m,tc,label='Cabin')
plt.xlabel('Time [min]'); plt.ylabel('Temperature [°C]'); plt.title('04E baseline transient')
plt.grid(True,alpha=.3); plt.legend(); plt.show()


In [ ]:
cop=np.array([r['cop'] for r in rows]); ph=np.array([r['p_high_bar'] for r in rows]); spd=np.array([r['compressor_speed_ratio'] for r in rows])
plt.plot(m,cop,label='COP'); plt.plot(m,spd,label='Compressor speed ratio')
plt.xlabel('Time [min]'); plt.title('Cycle/control response'); plt.grid(True,alpha=.3); plt.legend(); plt.show()


# 04F — Fault diagnosis
Compare generic proxy faults using the same model and KPIs. These are educational signatures, not OEM thresholds.


In [ ]:
summaries=[]
for name in FAULTS:
    summaries.append(summarize(name,run_fault_case(name,minutes=30)))
summaries


In [ ]:
names=[s['case'] for s in summaries]
peak_b=[s['peak_battery_c'] for s in summaries]
plt.figure(figsize=(10,4.5)); plt.bar(names,peak_b); plt.xticks(rotation=30,ha='right')
plt.ylabel('Peak battery temperature [°C]'); plt.title('04F fault comparison'); plt.tight_layout(); plt.show()


# 04G — Verification, calibration and literature benchmark
Verification is not validation. Calibration is also not validation.

Published benchmark: Periyasamy et al. (2026), *Asia-Pacific Journal of Chemical Engineering*, DOI 10.1002/apj.70202. The paper reports R134a experimental COP = 3.12 at evaporating −15 °C / condensing 40 °C with COP uncertainty ±0.08.

A single literature point is used here only as a transparent benchmark/calibration exercise. It is insufficient to claim full experimental validation.


In [ ]:
verification_checks(), compare_to_literature()


In [ ]:
eta=calibrate_efficiency_to_point()
eta, compare_to_literature(isentropic_efficiency=eta)


## 04G trust boundary
A defensible statement is: **the implementation is verified using conservation/trend checks and benchmarked against a published point; full validation requires multiple independent operating points with matched boundary conditions and measured quantities.**


# Engineering conclusion
Project 04 now spans system energy balance → refrigerant states → compressor envelope → heat exchangers → physical loads → transient plant/controls → fault diagnosis → evidence discipline.

The next major extension is **04H: 1D↔3D coupling**, where a cold-plate CFD/CHT model returns pressure-drop and thermal-performance maps to this plant model.
